# TP - DLA - Privacy defense (III) - use of OPACUS

Small example of the use of OPACUS for learning (under DP mechanism) a deep model

In [22]:
# ======================================================
# Differentially Private Deep Learning in ~20 lines
# ======================================================

!pip install --quiet torch torchvision opacus

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from opacus import PrivacyEngine

# -------------------------------
# 1. Data
# -------------------------------
transform = transforms.Compose([transforms.ToTensor()])
train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=20, shuffle=True)

# -------------------------------
# 2. Model
# -------------------------------
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28*28, 128)
        self.fc2 = nn.Linear(128, 10)
    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

model = Net()
optimizer = optim.SGD(model.parameters(), lr=0.1)
criterion = nn.CrossEntropyLoss()

# -------------------------------
# 3. Make optimizer private
# -------------------------------
privacy_engine = PrivacyEngine()
model, optimizer, train_loader = privacy_engine.make_private(
    module=model,
    optimizer=optimizer,
    data_loader=train_loader,
    noise_multiplier=1.2,   # adjust for stronger/weaker DP
    max_grad_norm=1.0,
)

# -------------------------------
# 4. Training loop (1 epoch demo)
# -------------------------------
model.train()
for batch_idx, (data, target) in enumerate(train_loader):
    optimizer.zero_grad()
    output = model(data)
    loss = criterion(output, target)
    loss.backward()
    optimizer.step()
    if batch_idx % 200 == 0:
        print(f"Train step {batch_idx}, Loss: {loss.item():.4f}")

# -------------------------------
# 5. Report privacy budget
# -------------------------------
epsilon = privacy_engine.get_epsilon(delta=1e-5)
print(f"Final privacy budget: ε = {epsilon:.2f}, δ = 1e-5")


Train step 0, Loss: 2.2853
Train step 200, Loss: 1.4679
Train step 400, Loss: 0.7668
Train step 600, Loss: 1.2663
Train step 800, Loss: 1.2208
Train step 1000, Loss: 0.8392
Train step 1200, Loss: 0.7118
Train step 1400, Loss: 1.0190
Train step 1600, Loss: 1.5081
Train step 1800, Loss: 2.0563
Train step 2000, Loss: 1.1101
Train step 2200, Loss: 2.5425
Train step 2400, Loss: 0.9120
Train step 2600, Loss: 0.5450
Train step 2800, Loss: 2.4889
Final privacy budget: ε = 0.07, δ = 1e-5
